In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# run_framework.py
# Medallion ETL Framework  —  L0 / L1 / L2
# ─────────────────────────────────────────────────────────────────────────────
# One GROUP_ID can drive MULTIPLE table loads.
# Parameters
#   GROUP_ID         : e.g. STU_ACTIVITY_L1  (required)
#   TARGET_LOAD_TABLE: run only this table   (optional — blank = run all)
#   ENVIRONMENT      : qa / prod             (optional — informational)
# VIEW rows are always skipped — handled by view_pipeline (DLT).
# ─────────────────────────────────────────────────────────────────────────────

In [0]:
# ── Widgets ───────────────────────────────────────────────────────────────────

dbutils.widgets.text("GROUP_ID",          "")
dbutils.widgets.text("TARGET_LOAD_TABLE", "")
dbutils.widgets.text("ENVIRONMENT",       "")

GROUP_ID          = dbutils.widgets.get("GROUP_ID").strip().upper()
TARGET_LOAD_TABLE = dbutils.widgets.get("TARGET_LOAD_TABLE").strip()
ENVIRONMENT       = dbutils.widgets.get("ENVIRONMENT").strip().lower() or "dev"

if not GROUP_ID:
    raise Exception(
        "❌  GROUP_ID is required.\n"
        "    Fix: set the GROUP_ID widget or job parameter.\n"
        "    Example: GROUP_ID = STU_ACTIVITY_L1"
    )

# Auto-detect layer from GROUP_ID suffix
RUN_LAYER = "ALL"
if   GROUP_ID.endswith("_L0"): RUN_LAYER = "L0"
elif GROUP_ID.endswith("_L1"): RUN_LAYER = "L1"
elif GROUP_ID.endswith("_L2"): RUN_LAYER = "L2"

print("=" * 60)
print(f"  GROUP_ID          : {GROUP_ID}")
print(f"  LAYER             : {RUN_LAYER}")
print(f"  TARGET_LOAD_TABLE : {TARGET_LOAD_TABLE or '(all tables)'}")
print(f"  ENVIRONMENT       : {ENVIRONMENT}")
print("=" * 60)

In [0]:
# ── Catalog auto-detect ───────────────────────────────────────────────────────

try:
    _cats   = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    CATALOG = "demo_catalog"   if "demo_catalog"   in _cats else \
              "hive_metastore" if "hive_metastore" in _cats else _cats[0]
except Exception:
    CATALOG = "hive_metastore"

print(f"  CATALOG           : {CATALOG}")

In [0]:
# ── Imports ───────────────────────────────────────────────────────────────────

import traceback
import requests
import io
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def banner(msg):
    print(f"\n{'─' * 60}\n  {msg}\n{'─' * 60}")


def err_block(context, exc, sql=None):
    lines = [
        "",
        f"  ┌── ERROR  {context}",
        f"  │   Type    : {type(exc).__name__}",
        f"  │   Message : {str(exc)[:350]}",
    ]
    if sql:
        lines.append(f"  │   SQL     : {sql[:200]}")
    lines.append("  └" + "─" * 50)
    return "\n".join(lines)


def read_source(url, fmt, delimiter=","):
    fmt = (fmt or "csv").strip().lower()
    url = (url or "").strip()

    if not url:
        raise ValueError("SOURCE_URL is empty in control table.")

    is_http = url.lower().startswith("http")

    if is_http:
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
        except requests.HTTPError as e:
            raise ValueError(f"HTTP {r.status_code} fetching source.\n  URL: {url}\n  {e}")

        raw = r.content
        if   fmt == "csv":                    return spark.createDataFrame(pd.read_csv(io.BytesIO(raw), sep=delimiter or ","))
        elif fmt == "json":                   return spark.createDataFrame(pd.read_json(io.BytesIO(raw)))
        elif fmt == "parquet":                return spark.createDataFrame(pd.read_parquet(io.BytesIO(raw)))
        elif fmt in ("excel", "xlsx", "xls"): return spark.createDataFrame(pd.read_excel(io.BytesIO(raw)))
        else: raise ValueError(f"Unsupported HTTP format: {fmt}. Use csv/json/parquet/excel.")
    else:
        if   fmt == "csv":     return spark.read.option("header","true").option("inferSchema","true").option("sep", delimiter or ",").csv(url)
        elif fmt == "json":    return spark.read.option("multiLine","true").json(url)
        elif fmt == "parquet": return spark.read.parquet(url)
        elif fmt == "delta":   return spark.read.format("delta").load(url)
        else: raise ValueError(f"Unsupported storage format: {fmt}. Use csv/json/parquet/delta.")


def ensure_schema(schema):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")


def qualify(query, src_schema, src_table):
    """Add catalog to bare schema/table references in SQL."""
    q = query
    if src_schema and f"{src_schema}." in q and f"{CATALOG}.{src_schema}." not in q:
        q = q.replace(f"{src_schema}.", f"{CATALOG}.{src_schema}.")
    if src_table and src_schema:
        for kw in ("FROM ", "JOIN "):
            bare = f"{kw}{src_table}"
            full = f"{kw}{CATALOG}.{src_schema}.{src_table}"
            if bare in q and full not in q:
                q = q.replace(bare, full)
    return q


def write_delta(df, schema, table, load_type, merge_keys=None):
    ensure_schema(schema)
    full = f"{CATALOG}.{schema}.{table}"
    lt   = (load_type or "FULL").strip().upper()

    if lt == "FULL":
        ( df.write.format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(full) )

    elif lt in ("INCREMENTAL", "APPEND"):
        ( df.write.format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .saveAsTable(full) )

    elif lt == "MERGE":
        if not merge_keys:
            raise ValueError(f"LOAD_TYPE=MERGE but MERGE_KEYS is empty for {full}.")
        keys       = [k.strip() for k in merge_keys.split(",")]
        match_cond = " AND ".join([f"tgt.{k} = src.{k}" for k in keys])
        tmp        = f"_tmp_{full.replace('.','_')}"
        df.createOrReplaceTempView(tmp)
        spark.sql(f"CREATE TABLE IF NOT EXISTS {full} USING DELTA AS SELECT * FROM {tmp} WHERE 1=0")
        upd = ", ".join([f"tgt.{c} = src.{c}" for c in df.columns if c not in keys])
        ins_c = ", ".join(df.columns)
        ins_v = ", ".join([f"src.{c}" for c in df.columns])
        spark.sql(f"""
            MERGE INTO {full} AS tgt
            USING {tmp} AS src
            ON    {match_cond}
            WHEN MATCHED     THEN UPDATE SET {upd}
            WHEN NOT MATCHED THEN INSERT ({ins_c}) VALUES ({ins_v})
        """)
    else:
        print(f"  ⚠️  Unknown LOAD_TYPE '{lt}' — defaulting to FULL")
        df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(full)

    return spark.table(full).count()


def log_audit(table_name, layer, status, rows, message, t0, t1):
    try:
        msg = str(message).replace("'", "''")[:500]
        spark.sql(f"""
            INSERT INTO {CATALOG}.admin.audit_log
            VALUES (
                '{GROUP_ID}', '{table_name}', '{layer}', '{status}',
                {rows}, '{msg}',
                '{t0.strftime('%Y-%m-%d %H:%M:%S')}',
                '{t1.strftime('%Y-%m-%d %H:%M:%S')}',
                current_timestamp()
            )
        """)
    except Exception as e:
        print(f"  ⚠️  Audit write failed (non-fatal): {e}")


In [0]:
# ── Validate header ───────────────────────────────────────────────────────────

try:
    hdr_count = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM   {CATALOG}.admin.data_flow_control_header
        WHERE  DATA_FLOW_GROUP_ID = '{GROUP_ID}'
        AND    IS_ACTIVE = 'Y'
    """).first()["n"]
except Exception as e:
    raise RuntimeError(
        f"❌  Cannot query control_header.\n"
        f"    Catalog : {CATALOG}\n"
        f"    Detail  : {e}\n"
        f"    Fix     : Run deployment pipeline first."
    )

if hdr_count == 0:
    raise Exception(
        f"❌  GROUP_ID '{GROUP_ID}' not found or IS_ACTIVE != 'Y'.\n"
        f"    Check  : SELECT * FROM {CATALOG}.admin.data_flow_control_header\n"
        f"             WHERE DATA_FLOW_GROUP_ID = '{GROUP_ID}'"
    )

print(f"\n  ✅  Header found for {GROUP_ID}")

In [0]:
# ── Core: process rows from one detail table ──────────────────────────────────

def process_layer(detail_table, layer, src_url_col="SOURCE_URL",
                  src_schema_col="SOURCE_OBJ_SCHEMA",
                  src_table_col="SOURCE_OBJ_NAME",
                  fmt_col="FILE_FORMAT", input_fmt_col="INPUT_FILE_FORMAT"):

    banner(f"{layer} — {detail_table}")

    # Build optional single-table filter
    tbl_filter = ""
    if TARGET_LOAD_TABLE:
        tbl_filter = f"AND TARGET_TABLE = '{TARGET_LOAD_TABLE}'"

    try:
        rows = spark.sql(f"""
            SELECT *
            FROM   {CATALOG}.admin.{detail_table}
            WHERE  DATA_FLOW_GROUP_ID = '{GROUP_ID}'
            AND    IS_ACTIVE          = 'Y'
            AND    UPPER(LOAD_TYPE)  != 'VIEW'
            {tbl_filter}
            ORDER BY TARGET_TABLE
        """).collect()
    except Exception as e:
        raise RuntimeError(
            f"❌  Cannot read {detail_table}.\n"
            f"    Detail : {e}\n"
            f"    Fix    : Run deployment pipeline to create {detail_table}."
        )

    if not rows:
        msg = f"No active rows in {detail_table} for GROUP_ID='{GROUP_ID}'"
        if TARGET_LOAD_TABLE:
            msg += f" / TARGET_LOAD_TABLE='{TARGET_LOAD_TABLE}'"
        print(f"  ℹ️  {msg} — skipping.")
        return

    print(f"  Tables to process : {len(rows)}")
    all_ok = True

    for row in rows:
        row_dict     = row.asDict()
        tgt_schema   = (row_dict.get("TARGET_SCHEMA")    or "").strip()
        tgt_table    = (row_dict.get("TARGET_TABLE")     or "").strip()
        load_type    = (row_dict.get("LOAD_TYPE")        or "FULL").strip().upper()
        merge_keys   = (row_dict.get("MERGE_KEYS")       or "").strip()
        trans_query  = (row_dict.get("TRANSFORMATION_QUERY") or "").strip()

        # L0 specific
        src_url    = (row_dict.get("SOURCE_URL")         or "").strip()
        src_schema = (row_dict.get("SOURCE_OBJ_SCHEMA")  or "").strip()
        src_table  = (row_dict.get("SOURCE_OBJ_NAME")    or "").strip()
        file_fmt   = (row_dict.get("INPUT_FILE_FORMAT")  or row_dict.get("FILE_FORMAT") or "csv").strip()
        delimiter  = (row_dict.get("DELIMETER")          or ",").strip()

        if not tgt_table or not tgt_schema:
            print("  ⚠️  Skipping row — TARGET_TABLE or TARGET_SCHEMA empty.")
            continue

        full_tgt = f"{CATALOG}.{tgt_schema}.{tgt_table}"
        print(f"\n  ▶  {full_tgt}")
        print(f"     LOAD_TYPE : {load_type}")

        t0     = datetime.now()
        status = "FAILED"
        n_rows = 0
        msg    = ""

        try:
            if layer == "L0":
                # ── Read from source ──────────────────────────────────
                print(f"     Source    : {src_url[:80] if src_url else 'N/A'}")
                df = read_source(src_url, file_fmt, delimiter)

            else:
                # ── L1 / L2 : apply TRANSFORMATION_QUERY ─────────────
                if trans_query:
                    sql = qualify(trans_query, src_schema, src_table)
                    print(f"     Query     : {sql[:80]}...")
                    df  = spark.sql(sql)
                else:
                    src_full = f"{CATALOG}.{src_schema}.{src_table}"
                    print(f"     Source    : {src_full}")
                    df = spark.table(src_full)

            # ── Add ETL audit columns ─────────────────────────────────
            df = ( df
                   .withColumn("_etl_group_id",  F.lit(GROUP_ID))
                   .withColumn("_etl_layer",      F.lit(layer))
                   .withColumn("_etl_env",        F.lit(ENVIRONMENT))
                   .withColumn("_etl_load_ts",    F.current_timestamp()) )

            # ── Write ─────────────────────────────────────────────────
            n_rows = write_delta(df, tgt_schema, tgt_table, load_type, merge_keys)
            status = "SUCCESS"
            msg    = f"{n_rows:,} rows loaded ({load_type})"
            print(f"     ✅  {msg}  →  {full_tgt}")

        except Exception as e:
            all_ok = False
            msg    = err_block(f"{layer}/{tgt_table}", e, trans_query or src_url)
            print(msg)

        finally:
            t1 = datetime.now()
            print(f"     ⏱  {round((t1 - t0).total_seconds(), 1)}s")
            log_audit(tgt_table, layer, status, n_rows, msg, t0, t1)

    if not all_ok:
        raise Exception(
            f"❌  {layer} had failures for GROUP_ID='{GROUP_ID}'.\n"
            f"    Check audit:\n"
            f"    SELECT * FROM {CATALOG}.admin.audit_log\n"
            f"    WHERE  DATA_FLOW_GROUP_ID = '{GROUP_ID}'\n"
            f"    AND    ETL_LAYER = '{layer}'\n"
            f"    ORDER BY LOAD_TS DESC"
        )

In [0]:
# ── Run layers ────────────────────────────────────────────────────────────────

pipeline_start = datetime.now()

if   RUN_LAYER == "L0": process_layer("data_flow_l0_detail", "L0")
elif RUN_LAYER == "L1": process_layer("data_flow_l1_detail", "L1")
elif RUN_LAYER == "L2": process_layer("data_flow_l2_detail", "L2")
else:
    process_layer("data_flow_l0_detail", "L0")
    process_layer("data_flow_l1_detail", "L1")
    process_layer("data_flow_l2_detail", "L2")

In [0]:
# ── Summary ───────────────────────────────────────────────────────────────────

total = round((datetime.now() - pipeline_start).total_seconds(), 1)

print("\n" + "=" * 60)
print("  ✅  PIPELINE COMPLETE")
print(f"  GROUP_ID          : {GROUP_ID}")
print(f"  LAYER             : {RUN_LAYER}")
print(f"  TARGET_LOAD_TABLE : {TARGET_LOAD_TABLE or '(all)'}")
print(f"  ENVIRONMENT       : {ENVIRONMENT}")
print(f"  CATALOG           : {CATALOG}")
print(f"  TOTAL DURATION    : {total}s")
print("─" * 60)
print(f"  Audit query:")
print(f"  SELECT * FROM {CATALOG}.admin.audit_log")
print(f"  WHERE  DATA_FLOW_GROUP_ID = '{GROUP_ID}'")
print(f"  ORDER BY LOAD_TS DESC")
print("=" * 60)